In [ ]:
from pathlib import Path
from itertools import combinations
import json
import re
import warnings
import sys

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


warnings.filterwarnings("ignore")

RANDOM_STATE = 42

In [ ]:
"""Reusable feature-selection and feature-reduction utilities for linear/meta models.

Purpose
-------
This module contains CPCV-safe feature-selection helpers that can be imported
from notebooks, training scripts, and hyperparameter-search code.

Available methods
-----------------
1. Correlation clustering
   `CorrelationClusterSelector` removes redundant highly correlated features by:
   - computing an absolute correlation matrix on the training features only,
   - building clusters of features where abs(correlation) >= threshold,
   - keeping one representative feature per cluster,
   - dropping the remaining features in that cluster.

2. PCA feature reduction
   `PCAFeatureReducer` converts numeric features into principal components by:
   - fitting imputation values on the training features only,
   - optionally standardising features using training-fold statistics only,
   - fitting PCA on the training fold only,
   - transforming validation/test folds with the fitted preprocessing + PCA.

Important CPCV note
-------------------
Fit selectors/reducers on the TRAIN fold only, then transform validation/test
folds using the fitted object. Do not fit on the full dataset before CPCV.

Examples
--------
Correlation clustering:

    from feature_selection_methods import CorrelationClusterSelector

    selector = CorrelationClusterSelector(
        corr_threshold=0.95,
        corr_method="spearman",
        selection_method="target_corr",
    )

    selector.fit(X_train, y_train)
    X_train_selected = selector.transform(X_train)
    X_test_selected = selector.transform(X_test)

PCA:

    from feature_selection_methods import PCAFeatureReducer

    pca_reducer = PCAFeatureReducer(
        n_components=0.95,
        standardize=True,
        component_prefix="pca",
    )

    pca_reducer.fit(X_train)
    X_train_pca = pca_reducer.transform(X_train)
    X_test_pca = pca_reducer.transform(X_test)

    print(pca_reducer.pca_summary_)
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Iterable, Literal

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


DEFAULT_EXCLUDE_COLUMNS = (
    "date",
    "instrument",
    "primary_signal",
    "triple_barrier_label",
    "metalabel",
    "target",
    "y",
)

CorrelationMethod = Literal["pearson", "spearman", "kendall"]
SelectionMethod = Literal["target_corr", "missing_then_variance", "variance", "first"]
ImputeStrategy = Literal["median", "mean", "zero"]


@dataclass
class CorrelationClusterResult:
    """Lightweight result object returned by `run_correlation_cluster_selection`."""

    selected_df: pd.DataFrame
    selected_features: list[str]
    dropped_features: list[str]
    cluster_summary: pd.DataFrame
    pair_summary: pd.DataFrame
    selector: "CorrelationClusterSelector"


@dataclass
class PCAFeatureReductionResult:
    """Lightweight result object returned by `run_pca_feature_reduction`."""

    transformed_df: pd.DataFrame
    component_columns: list[str]
    explained_variance_summary: pd.DataFrame
    reducer: "PCAFeatureReducer"


def _normalise_columns(columns: Iterable[str] | None) -> list[str]:
    """Convert column input to a clean list while preserving order."""

    if columns is None:
        return []
    return list(dict.fromkeys(columns))


def infer_numeric_feature_columns(
    df: pd.DataFrame,
    exclude_columns: Iterable[str] = DEFAULT_EXCLUDE_COLUMNS,
) -> list[str]:
    """Infer numeric feature columns, excluding ID/target/signal columns."""

    excluded = set(exclude_columns)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    return [col for col in numeric_cols if col not in excluded]


def _coerce_y_to_series(y: pd.Series | np.ndarray | list | None, index: pd.Index) -> pd.Series | None:
    """Return y as a numeric Series aligned to X's index where possible."""

    if y is None:
        return None

    if isinstance(y, pd.Series):
        if y.index.equals(index):
            y_series = y.copy()
        elif len(y) == len(index):
            y_series = pd.Series(y.to_numpy(), index=index, name=y.name)
        else:
            y_series = y.reindex(index)
    else:
        if len(y) != len(index):
            raise ValueError("y must have the same length as X if it is not a pandas Series.")
        y_series = pd.Series(y, index=index, name="target")

    return pd.to_numeric(y_series, errors="coerce")


def _target_correlation_scores(
    X: pd.DataFrame,
    y: pd.Series,
    method: CorrelationMethod,
) -> pd.Series:
    """Compute abs(feature-target correlation), handling constants safely."""

    scores: dict[str, float] = {}
    y_name = y.name if y.name is not None else "target"

    for col in X.columns:
        pair = pd.concat([X[col], y.rename(y_name)], axis=1).dropna()
        if pair.shape[0] < 3:
            scores[col] = np.nan
            continue
        if pair.iloc[:, 0].nunique(dropna=True) <= 1:
            scores[col] = np.nan
            continue
        if pair.iloc[:, 1].nunique(dropna=True) <= 1:
            scores[col] = np.nan
            continue

        corr_value = pair.iloc[:, 0].corr(pair.iloc[:, 1], method=method)
        scores[col] = abs(corr_value) if pd.notna(corr_value) else np.nan

    return pd.Series(scores, name="target_abs_corr")


def _high_correlation_pairs(
    corr_matrix: pd.DataFrame,
    threshold: float,
) -> pd.DataFrame:
    """Return all upper-triangle feature pairs with abs corr >= threshold."""

    if corr_matrix.empty:
        return pd.DataFrame(columns=["feature_1", "feature_2", "abs_corr"])

    corr_values = corr_matrix.to_numpy(dtype=float)
    upper_mask = np.triu(np.ones(corr_values.shape, dtype=bool), k=1)
    high_mask = upper_mask & np.isfinite(corr_values) & (corr_values >= threshold)
    row_idx, col_idx = np.where(high_mask)

    pairs = pd.DataFrame(
        {
            "feature_1": corr_matrix.index[row_idx],
            "feature_2": corr_matrix.columns[col_idx],
            "abs_corr": corr_values[row_idx, col_idx],
        }
    )

    if pairs.empty:
        return pairs

    return pairs.sort_values("abs_corr", ascending=False).reset_index(drop=True)


def _connected_components(
    features: list[str],
    pair_summary: pd.DataFrame,
) -> list[list[str]]:
    """Build correlation clusters from high-correlation edges."""

    order = {feature: position for position, feature in enumerate(features)}
    adjacency = {feature: set() for feature in features}

    for row in pair_summary.itertuples(index=False):
        f1 = row.feature_1
        f2 = row.feature_2
        adjacency[f1].add(f2)
        adjacency[f2].add(f1)

    clusters: list[list[str]] = []
    visited: set[str] = set()

    for feature in features:
        if feature in visited:
            continue

        stack = [feature]
        component: list[str] = []
        visited.add(feature)

        while stack:
            current = stack.pop()
            component.append(current)

            for neighbour in adjacency[current]:
                if neighbour not in visited:
                    visited.add(neighbour)
                    stack.append(neighbour)

        if len(component) > 1:
            clusters.append(sorted(component, key=lambda col: order[col]))

    return clusters


def _choose_representative(
    cluster: list[str],
    X: pd.DataFrame,
    selection_method: SelectionMethod,
    original_order: dict[str, int],
    target_scores: pd.Series | None = None,
) -> tuple[str, pd.DataFrame]:
    """Choose one representative feature from a correlation cluster."""

    stats = pd.DataFrame(index=cluster)
    stats["feature"] = cluster
    stats["original_order"] = [original_order[col] for col in cluster]
    stats["missing_rate"] = X[cluster].isna().mean()
    stats["n_unique"] = X[cluster].nunique(dropna=True)
    stats["variance"] = X[cluster].var(skipna=True)

    if target_scores is not None:
        stats["target_abs_corr"] = target_scores.reindex(cluster)
    else:
        stats["target_abs_corr"] = np.nan

    # If target correlation is requested but unavailable, fall back safely.
    effective_method = selection_method
    if selection_method == "target_corr" and stats["target_abs_corr"].notna().sum() == 0:
        effective_method = "missing_then_variance"

    if effective_method == "target_corr":
        ranked = stats.assign(
            target_abs_corr_for_sort=stats["target_abs_corr"].fillna(-np.inf),
            variance_for_sort=stats["variance"].fillna(-np.inf),
        ).sort_values(
            by=[
                "target_abs_corr_for_sort",
                "missing_rate",
                "n_unique",
                "variance_for_sort",
                "original_order",
            ],
            ascending=[False, True, False, False, True],
        )
    elif effective_method == "missing_then_variance":
        ranked = stats.assign(
            variance_for_sort=stats["variance"].fillna(-np.inf),
        ).sort_values(
            by=["missing_rate", "n_unique", "variance_for_sort", "original_order"],
            ascending=[True, False, False, True],
        )
    elif effective_method == "variance":
        ranked = stats.assign(
            variance_for_sort=stats["variance"].fillna(-np.inf),
        ).sort_values(
            by=["variance_for_sort", "missing_rate", "n_unique", "original_order"],
            ascending=[False, True, False, True],
        )
    elif effective_method == "first":
        ranked = stats.sort_values("original_order", ascending=True)
    else:
        raise ValueError(
            "selection_method must be one of "
            "'target_corr', 'missing_then_variance', 'variance', or 'first'."
        )

    representative = str(ranked.iloc[0]["feature"])
    return representative, stats.reset_index(drop=True)


class CorrelationClusterSelector:
    """Remove redundant features by keeping one feature per correlation cluster.

    Parameters
    ----------
    corr_threshold:
        Absolute correlation threshold used to connect two features into the
        same cluster. A value of 0.95 is a common starting point.
    corr_method:
        Correlation method passed to pandas. Spearman is often sensible for
        financial features because it is rank-based and less sensitive to
        extreme values than Pearson.
    selection_method:
        How to choose the representative feature from each cluster.
        - "target_corr": keep the feature with strongest abs correlation to y.
        - "missing_then_variance": prefer fewer missing values, then more unique
          values, then larger variance.
        - "variance": keep the highest-variance feature.
        - "first": keep the first feature in the original feature order.
    exclude_columns:
        Columns not to treat as candidate features when feature_columns is None.
    min_periods:
        Minimum number of overlapping observations required for a correlation.
    """

    def __init__(
        self,
        corr_threshold: float = 0.95,
        corr_method: CorrelationMethod = "spearman",
        selection_method: SelectionMethod = "target_corr",
        exclude_columns: Iterable[str] = DEFAULT_EXCLUDE_COLUMNS,
        min_periods: int = 20,
    ) -> None:
        if not 0 < corr_threshold < 1:
            raise ValueError("corr_threshold must be between 0 and 1.")
        if corr_method not in {"pearson", "spearman", "kendall"}:
            raise ValueError("corr_method must be 'pearson', 'spearman', or 'kendall'.")

        self.corr_threshold = corr_threshold
        self.corr_method = corr_method
        self.selection_method = selection_method
        self.exclude_columns = tuple(exclude_columns)
        self.min_periods = min_periods

    def fit(
        self,
        X: pd.DataFrame,
        y: pd.Series | np.ndarray | list | None = None,
        feature_columns: Iterable[str] | None = None,
    ) -> "CorrelationClusterSelector":
        """Fit the correlation cluster selector on training data only."""

        if not isinstance(X, pd.DataFrame):
            raise TypeError("X must be a pandas DataFrame.")

        if feature_columns is None:
            feature_cols = infer_numeric_feature_columns(X, self.exclude_columns)
        else:
            feature_cols = _normalise_columns(feature_columns)

        missing_cols = sorted(set(feature_cols) - set(X.columns))
        if missing_cols:
            raise ValueError(f"feature_columns not found in X: {missing_cols}")

        X_numeric = X[feature_cols].apply(pd.to_numeric, errors="coerce")
        feature_cols = X_numeric.columns.tolist()
        original_order = {feature: position for position, feature in enumerate(feature_cols)}

        y_series = _coerce_y_to_series(y, X.index)
        if self.selection_method == "target_corr" and y_series is not None:
            target_scores = _target_correlation_scores(X_numeric, y_series, self.corr_method)
        else:
            target_scores = None

        corr_matrix = X_numeric.corr(method=self.corr_method, min_periods=self.min_periods).abs()
        pair_summary = _high_correlation_pairs(corr_matrix, self.corr_threshold)
        clusters = _connected_components(feature_cols, pair_summary)

        selected_features = set(feature_cols)
        dropped_features: list[str] = []
        cluster_rows: list[dict[str, object]] = []

        for cluster_id, cluster in enumerate(clusters, start=1):
            representative, stats = _choose_representative(
                cluster=cluster,
                X=X_numeric,
                selection_method=self.selection_method,
                original_order=original_order,
                target_scores=target_scores,
            )

            cluster_drops = [feature for feature in cluster if feature != representative]
            selected_features.difference_update(cluster_drops)
            dropped_features.extend(cluster_drops)

            cluster_corr = corr_matrix.loc[cluster, cluster].copy()
            cluster_corr_values = cluster_corr.where(
                np.triu(np.ones(cluster_corr.shape, dtype=bool), k=1)
            ).stack()

            cluster_rows.append(
                {
                    "cluster_id": cluster_id,
                    "n_features": len(cluster),
                    "representative_feature": representative,
                    "dropped_features": cluster_drops,
                    "cluster_features": cluster,
                    "max_abs_corr_in_cluster": (
                        float(cluster_corr_values.max()) if not cluster_corr_values.empty else np.nan
                    ),
                    "mean_abs_corr_in_cluster": (
                        float(cluster_corr_values.mean()) if not cluster_corr_values.empty else np.nan
                    ),
                    "selection_method": self.selection_method,
                }
            )

        self.feature_columns_ = feature_cols
        self.selected_features_ = [feature for feature in feature_cols if feature in selected_features]
        self.dropped_features_ = [feature for feature in feature_cols if feature in set(dropped_features)]
        self.corr_matrix_ = corr_matrix
        self.pair_summary_ = pair_summary
        self.cluster_summary_ = pd.DataFrame(cluster_rows)
        self.target_scores_ = target_scores
        self.is_fitted_ = True

        return self

    def transform(
        self,
        X: pd.DataFrame,
        keep_columns: Iterable[str] | None = None,
    ) -> pd.DataFrame:
        """Return X with only selected features, plus optional ID/target columns."""

        if not getattr(self, "is_fitted_", False):
            raise RuntimeError("Call fit before transform.")
        if not isinstance(X, pd.DataFrame):
            raise TypeError("X must be a pandas DataFrame.")

        keep_cols = [col for col in _normalise_columns(keep_columns) if col in X.columns]
        missing_selected = [col for col in self.selected_features_ if col not in X.columns]
        if missing_selected:
            raise ValueError(f"Selected features missing from X: {missing_selected}")

        output_cols = keep_cols + [col for col in self.selected_features_ if col not in keep_cols]
        return X.loc[:, output_cols].copy()

    def fit_transform(
        self,
        X: pd.DataFrame,
        y: pd.Series | np.ndarray | list | None = None,
        feature_columns: Iterable[str] | None = None,
        keep_columns: Iterable[str] | None = None,
    ) -> pd.DataFrame:
        """Fit on X/y and return the selected version of X."""

        return self.fit(X, y=y, feature_columns=feature_columns).transform(
            X,
            keep_columns=keep_columns,
        )

    def get_support(self) -> pd.Series:
        """Boolean support mask indexed by the fitted feature columns."""

        if not getattr(self, "is_fitted_", False):
            raise RuntimeError("Call fit before get_support.")

        selected = set(self.selected_features_)
        return pd.Series(
            [feature in selected for feature in self.feature_columns_],
            index=self.feature_columns_,
            name="selected",
        )


class PCAFeatureReducer:
    """Reduce numeric features into principal components in a CPCV-safe way.

    Parameters
    ----------
    n_components:
        Number of components to keep. This follows sklearn PCA behaviour:
        - int: exact number of components, e.g. 10
        - float between 0 and 1: keep enough components to explain this fraction
          of variance, e.g. 0.95
        - None: keep all possible components
    standardize:
        If True, fit a StandardScaler on the training fold before PCA. This is
        usually recommended when features have different units/scales.
    impute_strategy:
        How to fill missing values before PCA, using training-fold statistics.
        Use "median" by default for financial features.
    component_prefix:
        Prefix for generated component column names.
    exclude_columns:
        Columns not to treat as candidate features when feature_columns is None.
    random_state:
        Random state passed to PCA. Mostly relevant for randomized solvers.
    svd_solver:
        Solver passed to sklearn PCA. "full" is deterministic and supports
        variance-ratio n_components such as 0.95.
    """

    def __init__(
        self,
        n_components: int | float | None = 0.95,
        standardize: bool = True,
        impute_strategy: ImputeStrategy = "median",
        component_prefix: str = "pca",
        exclude_columns: Iterable[str] = DEFAULT_EXCLUDE_COLUMNS,
        random_state: int | None = 42,
        svd_solver: str = "full",
    ) -> None:
        if isinstance(n_components, float) and not 0 < n_components <= 1:
            raise ValueError("If n_components is a float, it must be in the interval (0, 1].")
        if isinstance(n_components, int) and n_components <= 0:
            raise ValueError("If n_components is an int, it must be positive.")
        if impute_strategy not in {"median", "mean", "zero"}:
            raise ValueError("impute_strategy must be 'median', 'mean', or 'zero'.")
        if not component_prefix:
            raise ValueError("component_prefix must be a non-empty string.")

        self.n_components = n_components
        self.standardize = standardize
        self.impute_strategy = impute_strategy
        self.component_prefix = component_prefix
        self.exclude_columns = tuple(exclude_columns)
        self.random_state = random_state
        self.svd_solver = svd_solver

    def _fit_imputer(self, X_numeric: pd.DataFrame) -> pd.Series:
        """Fit per-feature imputation values using training data only."""

        if self.impute_strategy == "median":
            values = X_numeric.median(skipna=True)
        elif self.impute_strategy == "mean":
            values = X_numeric.mean(skipna=True)
        elif self.impute_strategy == "zero":
            values = pd.Series(0.0, index=X_numeric.columns)
        else:
            raise ValueError("Unsupported impute_strategy.")

        # A column that is entirely NaN will have NaN median/mean. Use 0 safely.
        return values.fillna(0.0)

    def _prepare_matrix_for_fit(self, X: pd.DataFrame, feature_columns: Iterable[str] | None) -> pd.DataFrame:
        """Select and numeric-coerce feature columns for fitting."""

        if not isinstance(X, pd.DataFrame):
            raise TypeError("X must be a pandas DataFrame.")

        if feature_columns is None:
            feature_cols = infer_numeric_feature_columns(X, self.exclude_columns)
        else:
            feature_cols = _normalise_columns(feature_columns)

        if not feature_cols:
            raise ValueError("No feature columns were provided/inferred for PCA.")

        missing_cols = sorted(set(feature_cols) - set(X.columns))
        if missing_cols:
            raise ValueError(f"feature_columns not found in X: {missing_cols}")

        return X[feature_cols].apply(pd.to_numeric, errors="coerce")

    def _prepare_matrix_for_transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """Apply fitted column order and imputation to new data."""

        if not getattr(self, "is_fitted_", False):
            raise RuntimeError("Call fit before transform.")
        if not isinstance(X, pd.DataFrame):
            raise TypeError("X must be a pandas DataFrame.")

        missing_features = [col for col in self.feature_columns_ if col not in X.columns]
        if missing_features:
            raise ValueError(f"PCA input features missing from X: {missing_features}")

        X_numeric = X[self.feature_columns_].apply(pd.to_numeric, errors="coerce")
        X_imputed = X_numeric.fillna(self.impute_values_)
        return X_imputed

    def fit(
        self,
        X: pd.DataFrame,
        y: pd.Series | np.ndarray | list | None = None,
        feature_columns: Iterable[str] | None = None,
    ) -> "PCAFeatureReducer":
        """Fit imputation, optional scaling, and PCA on training data only.

        The y argument is accepted for API symmetry with selectors but is not
        used, because PCA is unsupervised.
        """

        del y  # PCA is unsupervised; this avoids accidental use of labels.

        X_numeric = self._prepare_matrix_for_fit(X, feature_columns=feature_columns)
        self.feature_columns_ = X_numeric.columns.tolist()
        self.impute_values_ = self._fit_imputer(X_numeric)
        X_imputed = X_numeric.fillna(self.impute_values_)

        if X_imputed.shape[0] < 2:
            raise ValueError("PCA requires at least two rows in the training data.")

        max_components = min(X_imputed.shape[0], X_imputed.shape[1])
        if isinstance(self.n_components, int) and self.n_components > max_components:
            raise ValueError(
                f"n_components={self.n_components} is too large. Maximum allowed is "
                f"min(n_rows, n_features)={max_components}."
            )

        if self.standardize:
            self.scaler_ = StandardScaler()
            X_for_pca = self.scaler_.fit_transform(X_imputed)
        else:
            self.scaler_ = None
            X_for_pca = X_imputed.to_numpy(dtype=float)

        self.pca_ = PCA(
            n_components=self.n_components,
            svd_solver=self.svd_solver,
            random_state=self.random_state,
        )
        self.pca_.fit(X_for_pca)

        n_components_fitted = int(self.pca_.n_components_)
        self.component_columns_ = [
            f"{self.component_prefix}_{component_number:03d}"
            for component_number in range(1, n_components_fitted + 1)
        ]

        explained = self.pca_.explained_variance_ratio_
        self.explained_variance_ratio_ = pd.Series(
            explained,
            index=self.component_columns_,
            name="explained_variance_ratio",
        )
        self.cumulative_explained_variance_ = self.explained_variance_ratio_.cumsum()
        self.pca_summary_ = pd.DataFrame(
            {
                "component": self.component_columns_,
                "explained_variance_ratio": self.explained_variance_ratio_.values,
                "cumulative_explained_variance": self.cumulative_explained_variance_.values,
                "singular_value": self.pca_.singular_values_,
            }
        )

        self.n_original_features_ = len(self.feature_columns_)
        self.n_components_fitted_ = n_components_fitted
        self.is_fitted_ = True
        return self

    def transform(
        self,
        X: pd.DataFrame,
        keep_columns: Iterable[str] | None = None,
    ) -> pd.DataFrame:
        """Return PCA component dataframe, plus optional ID/target columns."""

        X_imputed = self._prepare_matrix_for_transform(X)

        if self.standardize:
            X_for_pca = self.scaler_.transform(X_imputed)
        else:
            X_for_pca = X_imputed.to_numpy(dtype=float)

        component_values = self.pca_.transform(X_for_pca)
        component_df = pd.DataFrame(
            component_values,
            columns=self.component_columns_,
            index=X.index,
        )

        keep_cols = [col for col in _normalise_columns(keep_columns) if col in X.columns]
        if keep_cols:
            return pd.concat([X.loc[:, keep_cols].copy(), component_df], axis=1)
        return component_df

    def fit_transform(
        self,
        X: pd.DataFrame,
        y: pd.Series | np.ndarray | list | None = None,
        feature_columns: Iterable[str] | None = None,
        keep_columns: Iterable[str] | None = None,
    ) -> pd.DataFrame:
        """Fit on X and return PCA-transformed X."""

        return self.fit(X, y=y, feature_columns=feature_columns).transform(
            X,
            keep_columns=keep_columns,
        )

    def get_component_loadings(self) -> pd.DataFrame:
        """Return PCA loadings with original features as rows and PCs as columns."""

        if not getattr(self, "is_fitted_", False):
            raise RuntimeError("Call fit before get_component_loadings.")

        return pd.DataFrame(
            self.pca_.components_.T,
            index=self.feature_columns_,
            columns=self.component_columns_,
        )

    def get_top_loadings(self, n: int = 10) -> pd.DataFrame:
        """Return the largest absolute loadings for each component.

        This is useful for interpreting which original features drive each PC.
        """

        if n <= 0:
            raise ValueError("n must be positive.")

        loadings = self.get_component_loadings()
        rows: list[dict[str, object]] = []
        for component in loadings.columns:
            top = loadings[component].abs().sort_values(ascending=False).head(n)
            for feature in top.index:
                rows.append(
                    {
                        "component": component,
                        "feature": feature,
                        "loading": loadings.loc[feature, component],
                        "abs_loading": abs(loadings.loc[feature, component]),
                    }
                )
        return pd.DataFrame(rows)


def run_correlation_cluster_selection(
    train_df: pd.DataFrame,
    target_col: str | None = None,
    feature_columns: Iterable[str] | None = None,
    keep_columns: Iterable[str] | None = None,
    corr_threshold: float = 0.95,
    corr_method: CorrelationMethod = "spearman",
    selection_method: SelectionMethod = "target_corr",
    exclude_columns: Iterable[str] = DEFAULT_EXCLUDE_COLUMNS,
    min_periods: int = 20,
) -> CorrelationClusterResult:
    """Convenience wrapper for fitting the selector and returning useful outputs.

    This should be called on the training fold only. Use the returned selector to
    transform validation/test folds.
    """

    if target_col is not None:
        if target_col not in train_df.columns:
            raise ValueError(f"target_col {target_col!r} not found in train_df.")
        y = train_df[target_col]
    else:
        y = None

    selector = CorrelationClusterSelector(
        corr_threshold=corr_threshold,
        corr_method=corr_method,
        selection_method=selection_method,
        exclude_columns=exclude_columns,
        min_periods=min_periods,
    )
    selected_df = selector.fit_transform(
        train_df,
        y=y,
        feature_columns=feature_columns,
        keep_columns=keep_columns,
    )

    return CorrelationClusterResult(
        selected_df=selected_df,
        selected_features=selector.selected_features_,
        dropped_features=selector.dropped_features_,
        cluster_summary=selector.cluster_summary_,
        pair_summary=selector.pair_summary_,
        selector=selector,
    )


def run_pca_feature_reduction(
    train_df: pd.DataFrame,
    feature_columns: Iterable[str] | None = None,
    keep_columns: Iterable[str] | None = None,
    n_components: int | float | None = 0.95,
    standardize: bool = True,
    impute_strategy: ImputeStrategy = "median",
    component_prefix: str = "pca",
    exclude_columns: Iterable[str] = DEFAULT_EXCLUDE_COLUMNS,
    random_state: int | None = 42,
    svd_solver: str = "full",
) -> PCAFeatureReductionResult:
    """Convenience wrapper for fitting PCA and returning useful outputs.

    This should be called on the training fold only. Use the returned reducer to
    transform validation/test folds.
    """

    reducer = PCAFeatureReducer(
        n_components=n_components,
        standardize=standardize,
        impute_strategy=impute_strategy,
        component_prefix=component_prefix,
        exclude_columns=exclude_columns,
        random_state=random_state,
        svd_solver=svd_solver,
    )
    transformed_df = reducer.fit_transform(
        train_df,
        feature_columns=feature_columns,
        keep_columns=keep_columns,
    )

    return PCAFeatureReductionResult(
        transformed_df=transformed_df,
        component_columns=reducer.component_columns_,
        explained_variance_summary=reducer.pca_summary_,
        reducer=reducer,
    )


__all__ = [
    "CorrelationClusterResult",
    "CorrelationClusterSelector",
    "DEFAULT_EXCLUDE_COLUMNS",
    "PCAFeatureReducer",
    "PCAFeatureReductionResult",
    "infer_numeric_feature_columns",
    "run_correlation_cluster_selection",
    "run_pca_feature_reduction",
]


In [ ]:
# Main ticker/model target
TICKER = "cl1s"

# Paths
TRIPLE_BARRIER_DIR = Path("../../data/features/triple_barrier")
FEATURE_SET_PATH = Path(f"../../data/features/clean_feature_set/{TICKER}_clean_feature_set.csv")

# Date settings
DATE_COL = "date"
INSTRUMENT_COL = "instrument"

# Label settings
# Change this if your triple-barrier CSV uses a different target column name.
TARGET_COL = "metalabel"

# Columns that should not be used as model features
NON_FEATURE_COLS = {
    # Identifiers / indexing
    "date",
    "instrument",

    # Target / label columns
    TARGET_COL,
    "target",
    "label",
    "y",
    "meta_label",
    "tb_label",
    "barrier_label",

    # Triple-barrier metadata / leakage-prone columns
    "num_days",
    "t1",
    "vertical_barrier_date",
    "barrier_touch_date",
    "event_end_date",
    "exit_date",
    "exit_price",
    "exit_return",
    "triple_barrier_return",
    "tb_return",
    "realised_return",
    "realized_return",
    "pnl",
    "profit",
    "barrier_touched",
    "first_barrier_touched",
    "hit_barrier",
    "pt",
    "sl",
    "pt_mult",
    "sl_mult",
    "volatility",
    "target_vol",
}

In [ ]:
def extract_num_days_from_filename(path: Path) -> int | None:
    """
    Tries to extract num_days from filenames such as:
    cl1s_tb_num_days_10.csv
    triple_barrier_cl1s_10d.csv
    cl1s_pt1_sl1_10.csv
    """
    filename = path.stem.lower()

    patterns = [
        r"num_days[_\-]?(\d+)",
        r"(\d+)d",
        r"horizon[_\-]?(\d+)",
    ]

    for pattern in patterns:
        match = re.search(pattern, filename)
        if match:
            return int(match.group(1))

    return None


def load_feature_set(path: Path, ticker: str) -> pd.DataFrame:
    """
    Loads the clean feature set for a specific ticker.
    """
    if not path.exists():
        raise FileNotFoundError(f"Feature set not found: {path}")

    features = pd.read_csv(path)
    features[DATE_COL] = pd.to_datetime(features[DATE_COL])

    if INSTRUMENT_COL in features.columns:
        features[INSTRUMENT_COL] = features[INSTRUMENT_COL].str.lower()
        features = features[features[INSTRUMENT_COL] == ticker.lower()].copy()

    features = features.sort_values(DATE_COL).reset_index(drop=True)

    return features


def load_triple_barrier_files(
    triple_barrier_dir: Path,
    ticker: str,
) -> dict[str, pd.DataFrame]:
    """
    Loads all triple-barrier CSV files for the given ticker.

    IMPORTANT:
    Only keeps merge keys + TARGET_COL.
    This prevents triple-barrier leakage columns from becoming model features.
    """
    if not triple_barrier_dir.exists():
        raise FileNotFoundError(f"Triple-barrier directory not found: {triple_barrier_dir}")

    csv_files = sorted(triple_barrier_dir.glob("*.csv"))

    ticker_files = [
        path for path in csv_files
        if ticker.lower() in path.name.lower()
    ]

    if len(ticker_files) == 0:
        raise FileNotFoundError(
            f"No triple-barrier CSV files found for ticker '{ticker}' in {triple_barrier_dir}"
        )

    tb_data = {}

    for path in ticker_files:
        df = pd.read_csv(path)
        df[DATE_COL] = pd.to_datetime(df[DATE_COL])

        if INSTRUMENT_COL in df.columns:
            df[INSTRUMENT_COL] = df[INSTRUMENT_COL].str.lower()
            df = df[df[INSTRUMENT_COL] == ticker.lower()].copy()

        # Get num_days only as metadata for CPCV embargo, not as a model column.
        if "num_days" in df.columns:
            num_days = int(pd.to_numeric(df["num_days"], errors="coerce").dropna().mode().iloc[0])
        else:
            num_days = extract_num_days_from_filename(path)

        # Find the label column.
        possible_label_cols = [TARGET_COL, "y"]
        label_col = next((col for col in possible_label_cols if col in df.columns), None)

        if label_col is None:
            raise ValueError(
                f"No metalabel/target column found in {path.name}. "
                f"Available columns: {list(df.columns)}"
            )

        # Keep ONLY merge keys + label.
        keep_cols = [DATE_COL]

        if INSTRUMENT_COL in df.columns:
            keep_cols.append(INSTRUMENT_COL)

        keep_cols.append(label_col)

        df = df[keep_cols].copy()

        # Standardise label name back to TARGET_COL, e.g. "metalabel".
        if label_col != TARGET_COL:
            df = df.rename(columns={label_col: TARGET_COL})

        df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")

        df = df.sort_values(DATE_COL).reset_index(drop=True)

        # Store num_days as dataframe metadata, not as a column.
        df.attrs["num_days"] = num_days

        tb_data[path.stem] = df

    return tb_data

In [ ]:
features = load_feature_set(FEATURE_SET_PATH, TICKER)
tb_configs = load_triple_barrier_files(TRIPLE_BARRIER_DIR, TICKER)

print(f"Loaded feature set for {TICKER}: {features.shape}")
print(f"Loaded {len(tb_configs)} triple-barrier configurations:")

for config_name, tb_df in tb_configs.items():
    print(
        f"  {config_name}: shape={tb_df.shape}, "
        f"columns={list(tb_df.columns)}, "
        f"num_days={tb_df.attrs.get('num_days')}, "
        f"target_counts={tb_df[TARGET_COL].value_counts(dropna=False).to_dict()}"
    )

display(features.head())

first_config_name = list(tb_configs.keys())[0]
display(tb_configs[first_config_name].head())

In [ ]:

# Clean out-of-sample period for local evaluation
# You can change these if your group decides on another split.
TEST_START_DATE = "2022-01-01"
TEST_END_DATE = "2022-06-30"

# For meta-labeling, the cleanest setup is usually to train only where the primary model made a trade.
# The meta-model learns: should we take this +1/-1 signal?
TRAIN_ON_NONZERO_SIGNALS_ONLY = True


def get_feature_cols(df: pd.DataFrame, non_feature_cols: set[str]) -> list[str]:
    """
    Selects model feature columns from the merged dataframe.
    Keeps primary_signal as a feature unless it is explicitly in NON_FEATURE_COLS.
    """
    feature_cols = [
        col for col in df.columns
        if col not in non_feature_cols
    ]

    # Keep only numeric columns because sklearn models need numeric X.
    numeric_feature_cols = [
        col for col in feature_cols
        if pd.api.types.is_numeric_dtype(df[col])
    ]

    dropped_non_numeric = sorted(set(feature_cols) - set(numeric_feature_cols))
    if dropped_non_numeric:
        print(f"Dropped non-numeric feature columns: {dropped_non_numeric}")

    return numeric_feature_cols


def merge_features_with_tb(
    features_df: pd.DataFrame,
    tb_df: pd.DataFrame,
    ticker: str,
    target_col: str = TARGET_COL,
) -> pd.DataFrame:
    """
    Merges the clean feature set with one triple-barrier config.

    Expected:
        features_df has date, instrument, features
        tb_df has date, instrument, metalabel, num_days, etc.

    Output:
        one merged dataframe with features + target + triple-barrier metadata.
    """
    features_clean = features_df.copy()
    tb_clean = tb_df.copy()

    features_clean[DATE_COL] = pd.to_datetime(features_clean[DATE_COL])
    tb_clean[DATE_COL] = pd.to_datetime(tb_clean[DATE_COL])

    if INSTRUMENT_COL in features_clean.columns:
        features_clean[INSTRUMENT_COL] = features_clean[INSTRUMENT_COL].str.lower()
        features_clean = features_clean[features_clean[INSTRUMENT_COL] == ticker.lower()].copy()

    if INSTRUMENT_COL in tb_clean.columns:
        tb_clean[INSTRUMENT_COL] = tb_clean[INSTRUMENT_COL].str.lower()
        tb_clean = tb_clean[tb_clean[INSTRUMENT_COL] == ticker.lower()].copy()

    # If earlier loading renamed metalabel to target, rename it back/check cleanly here.
    if "target" in tb_clean.columns and target_col not in tb_clean.columns:
        tb_clean = tb_clean.rename(columns={"target": target_col})

    if target_col not in tb_clean.columns:
        raise ValueError(
            f"Target column '{target_col}' not found in triple-barrier dataframe. "
            f"Available columns: {list(tb_clean.columns)}"
        )

    merge_keys = [DATE_COL]
    if INSTRUMENT_COL in features_clean.columns and INSTRUMENT_COL in tb_clean.columns:
        merge_keys.append(INSTRUMENT_COL)

    merged = features_clean.merge(
        tb_clean,
        on=merge_keys,
        how="inner",
        suffixes=("", "_tb"),
    )

    merged = merged.sort_values(DATE_COL).reset_index(drop=True)

    return merged


def clean_model_dataframe(
    merged_df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str = TARGET_COL,
    train_on_nonzero_signals_only: bool = TRAIN_ON_NONZERO_SIGNALS_ONLY,
) -> pd.DataFrame:
    """
    Cleans the merged dataframe before splitting:
    - optionally removes primary_signal == 0 rows
    - removes rows with missing target
    - replaces inf with NaN
    - drops rows with missing X or y
    """
    df = merged_df.copy()

    if train_on_nonzero_signals_only and "primary_signal" in df.columns:
        df = df[df["primary_signal"] != 0].copy()

    df = df.replace([np.inf, -np.inf], np.nan)

    required_cols = feature_cols + [target_col, DATE_COL]
    df = df.dropna(subset=required_cols).copy()

    # Make sure target is integer binary 0/1
    df[target_col] = df[target_col].astype(int)

    invalid_targets = sorted(set(df[target_col].unique()) - {0, 1})
    if invalid_targets:
        raise ValueError(
            f"Target column '{target_col}' must be binary 0/1. "
            f"Found invalid values: {invalid_targets}"
        )

    df = df.sort_values(DATE_COL).reset_index(drop=True)

    return df


def chronological_train_test_split(
    df: pd.DataFrame,
    test_start_date: str = TEST_START_DATE,
    test_end_date: str = TEST_END_DATE,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Chronological train/test split.

    Train:
        date < TEST_START_DATE

    Test:
        TEST_START_DATE <= date <= TEST_END_DATE
    """
    test_start_date = pd.to_datetime(test_start_date)
    test_end_date = pd.to_datetime(test_end_date)

    train_df = df[df[DATE_COL] < test_start_date].copy()

    test_df = df[
        (df[DATE_COL] >= test_start_date)
        & (df[DATE_COL] <= test_end_date)
    ].copy()

    return train_df, test_df


def build_xy(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str = TARGET_COL,
):
    """
    Builds X/y arrays for sklearn.
    """
    X_train = train_df[feature_cols].copy()
    y_train = train_df[target_col].copy()

    X_test = test_df[feature_cols].copy()
    y_test = test_df[target_col].copy()

    return X_train, y_train, X_test, y_test

In [ ]:
# =========================
# 6. Build model datasets for every triple-barrier config
# =========================

datasets_by_config = {}

for config_name, tb_df in tb_configs.items():
    print("=" * 80)
    print(f"Building dataset for triple-barrier config: {config_name}")

    merged_df = merge_features_with_tb(
        features_df=features,
        tb_df=tb_df,
        ticker=TICKER,
        target_col=TARGET_COL,
    )

    feature_cols = get_feature_cols(
        df=merged_df,
        non_feature_cols=NON_FEATURE_COLS,
    )

    model_df = clean_model_dataframe(
        merged_df=merged_df,
        feature_cols=feature_cols,
        target_col=TARGET_COL,
        train_on_nonzero_signals_only=TRAIN_ON_NONZERO_SIGNALS_ONLY,
    )

    train_df, test_df = chronological_train_test_split(
        df=model_df,
        test_start_date=TEST_START_DATE,
        test_end_date=TEST_END_DATE,
    )

    X_train, y_train, X_test, y_test = build_xy(
        train_df=train_df,
        test_df=test_df,
        feature_cols=feature_cols,
        target_col=TARGET_COL,
    )

    datasets_by_config[config_name] = {
        "merged_df": merged_df,
        "model_df": model_df,
        "train_df": train_df,
        "test_df": test_df,
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test,
        "feature_cols": feature_cols,
        "num_days": int(model_df["num_days"].mode().iloc[0]) if "num_days" in model_df.columns else None,
    }

    print(f"Merged shape:     {merged_df.shape}")
    print(f"Model shape:      {model_df.shape}")
    print(f"Train shape:      {train_df.shape}")
    print(f"Test shape:       {test_df.shape}")
    print(f"X_train shape:    {X_train.shape}")
    print(f"X_test shape:     {X_test.shape}")
    print(f"Num features:     {len(feature_cols)}")
    print(f"Train target dist:{y_train.value_counts(normalize=True).round(3).to_dict()}")
    print(f"Test target dist: {y_test.value_counts(normalize=True).round(3).to_dict()}")
    print(f"num_days:         {datasets_by_config[config_name]['num_days']}")

In [ ]:
# =========================
# 7. Sanity check one config
# =========================

first_config_name = list(datasets_by_config.keys())[0]
data = datasets_by_config[first_config_name]

print(f"Using config: {first_config_name}")

display(data["train_df"].head())
display(data["test_df"].head())

print("Feature columns:")
print(data["feature_cols"])

print("\nTrain date range:")
print(data["train_df"][DATE_COL].min(), "to", data["train_df"][DATE_COL].max())

print("\nTest date range:")
print(data["test_df"][DATE_COL].min(), "to", data["test_df"][DATE_COL].max())

print("\nAny NaNs in X_train?")
print(data["X_train"].isna().sum().sort_values(ascending=False).head(10))

print("\nAny NaNs in X_test?")
print(data["X_test"].isna().sum().sort_values(ascending=False).head(10))

In [ ]:
RF_CONFIGS = [
    {
        "model_name": "rf_baseline",
        "model_params": {
            "n_estimators": 300,
            "max_depth": None,
            "min_samples_leaf": 5,
            "min_samples_split": 10,
            "max_features": "sqrt",
            "class_weight": "balanced",
            "bootstrap": True,
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
        },
    },
    {
        "model_name": "rf_shallow",
        "model_params": {
            "n_estimators": 300,
            "max_depth": 4,
            "min_samples_leaf": 10,
            "min_samples_split": 20,
            "max_features": "sqrt",
            "class_weight": "balanced",
            "bootstrap": True,
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
        },
    },
    {
        "model_name": "rf_medium_depth",
        "model_params": {
            "n_estimators": 500,
            "max_depth": 8,
            "min_samples_leaf": 8,
            "min_samples_split": 20,
            "max_features": "sqrt",
            "class_weight": "balanced",
            "bootstrap": True,
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
        },
    },
    {
        "model_name": "rf_conservative",
        "model_params": {
            "n_estimators": 500,
            "max_depth": 5,
            "min_samples_leaf": 20,
            "min_samples_split": 40,
            "max_features": 0.5,
            "class_weight": "balanced",
            "bootstrap": True,
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
        },
    },
]

# =========================
# 8B. Feature selection / reduction configs
# =========================

PROTECTED_FEATURES = [
    "primary_signal",
]

FEATURE_PROCESSING_CONFIGS = [
    {
        "feature_method": "corr_cluster",
        "feature_params": {
            "corr_threshold": 0.95,
            "corr_method": "spearman",
            "selection_method": "target_corr",
            "min_periods": 20,
        },
    },
        {
        "feature_method": "corr_cluster",
        "feature_params": {
            "corr_threshold": 0.90,
            "corr_method": "spearman",
            "selection_method": "target_corr",
            "min_periods": 20,
        },
    },
    {
        "feature_method": "pca",
        "feature_params": {
            "n_components": 0.95,
            "standardize": True,
            "impute_strategy": "median",
            "component_prefix": "pca",
            "random_state": RANDOM_STATE,
            "svd_solver": "full",
        },
    },
]


def make_random_forest(model_params: dict) -> RandomForestClassifier:
    """
    Creates a Random Forest model from a parameter dictionary.
    """
    return RandomForestClassifier(**model_params)

In [ ]:
CPCV_N_BLOCKS = 10
CPCV_N_TEST_BLOCKS = 2


def make_time_blocks(n_samples: int, n_blocks: int) -> list[np.ndarray]:
    """
    Splits row positions into chronological blocks.
    """
    indices = np.arange(n_samples)
    blocks = np.array_split(indices, n_blocks)
    return [block for block in blocks if len(block) > 0]


def get_label_end_series(
    df: pd.DataFrame,
    fallback_num_days: int | None = None,
) -> pd.Series:
    """
    Returns the label end date for purging.

    Prefer timeout_date because it is the maximum possible information horizon.
    This is conservative and directly linked to num_days.
    """
    if "timeout_date" in df.columns:
        return pd.to_datetime(df["timeout_date"])

    if "touch_date" in df.columns:
        return pd.to_datetime(df["touch_date"])

    if fallback_num_days is None:
        raise ValueError(
            "No timeout_date/touch_date found and fallback_num_days was not provided."
        )

    return pd.to_datetime(df[DATE_COL]) + pd.Timedelta(days=fallback_num_days)


def generate_cpcv_splits(
    train_df: pd.DataFrame,
    n_blocks: int = CPCV_N_BLOCKS,
    n_test_blocks: int = CPCV_N_TEST_BLOCKS,
    embargo_size: int | None = None,
    date_col: str = DATE_COL,
):
    """
    Generates CPCV train/test index splits with purging and embargo.

    Purging:
        Remove training samples whose label interval overlaps the test interval.

    Embargo:
        Remove the next embargo_size rows after each test block.
        For this coursework, embargo_size should usually be num_days.
    """
    df = train_df.sort_values(date_col).reset_index(drop=True).copy()

    n_samples = len(df)

    if n_samples < n_blocks:
        raise ValueError(
            f"Not enough samples for CPCV: n_samples={n_samples}, n_blocks={n_blocks}"
        )

    if embargo_size is None:
        if "num_days" in df.columns:
            embargo_size = int(df["num_days"].mode().iloc[0])
        else:
            embargo_size = 0

    blocks = make_time_blocks(n_samples=n_samples, n_blocks=n_blocks)

    event_start = pd.to_datetime(df[date_col]).reset_index(drop=True)
    event_end = get_label_end_series(df, fallback_num_days=embargo_size).reset_index(drop=True)

    split_id = 0

    for test_block_ids in combinations(range(len(blocks)), n_test_blocks):
        split_id += 1

        test_idx = np.concatenate([blocks[block_id] for block_id in test_block_ids])
        test_idx = np.sort(test_idx)

        test_mask = np.zeros(n_samples, dtype=bool)
        test_mask[test_idx] = True

        train_mask = ~test_mask

        # Purge and embargo around each selected test block separately
        for block_id in test_block_ids:
            block_idx = blocks[block_id]

            test_block_start = event_start.iloc[block_idx].min()
            test_block_end = event_end.iloc[block_idx].max()

            # Purge train rows whose label interval overlaps this test block interval
            overlap_mask = (
                (event_start <= test_block_end)
                & (event_end >= test_block_start)
            ).to_numpy()

            train_mask[overlap_mask] = False

            # Embargo rows immediately after the test block
            block_last_pos = int(block_idx.max())
            embargo_start_pos = block_last_pos + 1
            embargo_end_pos = min(block_last_pos + embargo_size, n_samples - 1)

            if embargo_start_pos <= embargo_end_pos:
                embargo_idx = np.arange(embargo_start_pos, embargo_end_pos + 1)
                train_mask[embargo_idx] = False

        train_idx = np.where(train_mask)[0]

        yield {
            "split_id": split_id,
            "test_block_ids": test_block_ids,
            "train_idx": train_idx,
            "test_idx": test_idx,
            "embargo_size": embargo_size,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
        }

In [ ]:
# =========================
# 10. Metrics helpers
# =========================

def get_positive_class_proba(model, X: pd.DataFrame) -> np.ndarray:
    """
    Returns P(y=1) from a fitted sklearn classifier.
    """
    proba = model.predict_proba(X)

    classes = list(model.classes_)

    if 1 not in classes:
        raise ValueError(f"Model was not trained with positive class 1. Classes: {classes}")

    positive_class_index = classes.index(1)

    return proba[:, positive_class_index]


def classification_metrics(
    y_true: pd.Series,
    y_proba: np.ndarray,
    threshold: float = 0.5,
) -> dict:
    """
    Computes classification metrics for one fold.
    """
    y_true = pd.Series(y_true).astype(int)
    y_pred = (y_proba >= threshold).astype(int)

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

    # AUC is only defined if both classes are present in y_true
    if y_true.nunique() == 2:
        metrics["auc"] = roc_auc_score(y_true, y_proba)
    else:
        metrics["auc"] = np.nan

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    metrics.update(
        {
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "tp": tp,
        }
    )

    return metrics

In [ ]:
# =========================
# 10B. CPCV-safe feature processing helpers
# =========================

def split_protected_and_reducible_features(
    feature_cols: list[str],
    protected_features: list[str] = PROTECTED_FEATURES,
) -> tuple[list[str], list[str]]:
    """
    Splits features into:
    - protected features: always kept raw, e.g. primary_signal
    - reducible features: eligible for correlation selection or PCA
    """
    protected = [col for col in protected_features if col in feature_cols]
    reducible = [col for col in feature_cols if col not in protected]

    return protected, reducible


def apply_feature_processing_for_fold(
    fold_train: pd.DataFrame,
    fold_test: pd.DataFrame,
    feature_cols: list[str],
    y_fold_train: pd.Series,
    feature_processing_config: dict,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str], dict]:
    """
    Fits feature selection/reduction on the CPCV TRAIN fold only,
    then transforms both TRAIN and TEST folds.

    This is the key anti-leakage step.
    """
    method = feature_processing_config["feature_method"]
    params = feature_processing_config.get("feature_params", {})

    protected_cols, reducible_cols = split_protected_and_reducible_features(
        feature_cols=feature_cols,
        protected_features=PROTECTED_FEATURES,
    )

    X_train_protected = fold_train[protected_cols].copy() if protected_cols else pd.DataFrame(index=fold_train.index)
    X_test_protected = fold_test[protected_cols].copy() if protected_cols else pd.DataFrame(index=fold_test.index)

    if method == "none":
        X_train_processed = fold_train[feature_cols].copy()
        X_test_processed = fold_test[feature_cols].copy()

        processing_info = {
            "feature_method": method,
            "n_original_features": len(feature_cols),
            "n_processed_features": len(feature_cols),
            "n_dropped_features": 0,
            "processor": None,
        }

        return X_train_processed, X_test_processed, feature_cols, processing_info

    elif method == "corr_cluster":
        selector = CorrelationClusterSelector(**params)

        # Fit ONLY on CPCV training fold
        selector.fit(
            X=fold_train[reducible_cols],
            y=y_fold_train,
            feature_columns=reducible_cols,
        )

        X_train_selected = selector.transform(fold_train[reducible_cols])
        X_test_selected = selector.transform(fold_test[reducible_cols])

        X_train_processed = pd.concat(
            [X_train_protected.reset_index(drop=True), X_train_selected.reset_index(drop=True)],
            axis=1,
        )

        X_test_processed = pd.concat(
            [X_test_protected.reset_index(drop=True), X_test_selected.reset_index(drop=True)],
            axis=1,
        )

        processed_feature_cols = protected_cols + selector.selected_features_

        processing_info = {
            "feature_method": method,
            "n_original_features": len(feature_cols),
            "n_reducible_features": len(reducible_cols),
            "n_protected_features": len(protected_cols),
            "n_processed_features": len(processed_feature_cols),
            "n_dropped_features": len(selector.dropped_features_),
            "selected_features": selector.selected_features_,
            "dropped_features": selector.dropped_features_,
            "processor": selector,
        }

        return X_train_processed, X_test_processed, processed_feature_cols, processing_info

    elif method == "pca":
        reducer = PCAFeatureReducer(**params)

        # Fit ONLY on CPCV training fold
        reducer.fit(
            X=fold_train[reducible_cols],
            feature_columns=reducible_cols,
        )

        X_train_pca = reducer.transform(fold_train[reducible_cols])
        X_test_pca = reducer.transform(fold_test[reducible_cols])

        X_train_processed = pd.concat(
            [X_train_protected.reset_index(drop=True), X_train_pca.reset_index(drop=True)],
            axis=1,
        )

        X_test_processed = pd.concat(
            [X_test_protected.reset_index(drop=True), X_test_pca.reset_index(drop=True)],
            axis=1,
        )

        processed_feature_cols = protected_cols + reducer.component_columns_

        processing_info = {
            "feature_method": method,
            "n_original_features": len(feature_cols),
            "n_reducible_features": len(reducible_cols),
            "n_protected_features": len(protected_cols),
            "n_processed_features": len(processed_feature_cols),
            "n_pca_components": len(reducer.component_columns_),
            "pca_explained_variance": (
                float(reducer.cumulative_explained_variance_.iloc[-1])
                if len(reducer.cumulative_explained_variance_) > 0
                else np.nan
            ),
            "processor": reducer,
        }

        return X_train_processed, X_test_processed, processed_feature_cols, processing_info

    else:
        raise ValueError(
            f"Unknown feature processing method: {method}. "
            "Expected one of: 'none', 'corr_cluster', 'pca'."
        )
# =========================
# 11. Run CPCV for one model/config
# =========================

# =========================
# 11. Run CPCV for one model/config with feature processing
# =========================

def run_cpcv_for_model(
    model,
    train_df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str = TARGET_COL,
    feature_processing_config: dict | None = None,
    n_blocks: int = CPCV_N_BLOCKS,
    n_test_blocks: int = CPCV_N_TEST_BLOCKS,
    embargo_size: int | None = None,
    threshold: float = 0.5,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Runs CPCV for one model configuration.

    Feature selection/reduction is fitted inside each CPCV fold
    using only the fold's training data, then applied to that fold's test data.
    """
    if feature_processing_config is None:
        feature_processing_config = {
            "feature_method": "none",
            "feature_params": {},
        }

    df = train_df.sort_values(DATE_COL).reset_index(drop=True).copy()

    fold_rows = []
    prediction_rows = []

    splits = list(
        generate_cpcv_splits(
            train_df=df,
            n_blocks=n_blocks,
            n_test_blocks=n_test_blocks,
            embargo_size=embargo_size,
            date_col=DATE_COL,
        )
    )

    for split in splits:
        split_id = split["split_id"]
        train_idx = split["train_idx"]
        test_idx = split["test_idx"]

        fold_train = df.iloc[train_idx].copy()
        fold_test = df.iloc[test_idx].copy()

        y_fold_train = fold_train[target_col].astype(int)
        y_fold_test = fold_test[target_col].astype(int)

        # Skip impossible folds
        if y_fold_train.nunique() < 2:
            print(f"Skipping split {split_id}: train fold has only one class.")
            continue

        # CPCV-safe feature selection/reduction:
        # fit on fold_train only, transform fold_test only.
        (
            X_fold_train,
            X_fold_test,
            processed_feature_cols,
            processing_info,
        ) = apply_feature_processing_for_fold(
            fold_train=fold_train,
            fold_test=fold_test,
            feature_cols=feature_cols,
            y_fold_train=y_fold_train,
            feature_processing_config=feature_processing_config,
        )

        fitted_model = clone(model)
        fitted_model.fit(X_fold_train, y_fold_train)

        y_proba = get_positive_class_proba(fitted_model, X_fold_test)

        metrics = classification_metrics(
            y_true=y_fold_test,
            y_proba=y_proba,
            threshold=threshold,
        )

        fold_rows.append(
            {
                "split_id": split_id,
                "test_block_ids": split["test_block_ids"],
                "n_train": split["n_train"],
                "n_test": split["n_test"],
                "embargo_size": split["embargo_size"],
                "feature_method": feature_processing_config["feature_method"],
                "n_original_features": processing_info.get("n_original_features", np.nan),
                "n_processed_features": processing_info.get("n_processed_features", np.nan),
                "n_dropped_features": processing_info.get("n_dropped_features", np.nan),
                "n_pca_components": processing_info.get("n_pca_components", np.nan),
                "pca_explained_variance": processing_info.get("pca_explained_variance", np.nan),
                **metrics,
            }
        )

        fold_predictions = fold_test[[DATE_COL, INSTRUMENT_COL]].copy()
        fold_predictions["split_id"] = split_id
        fold_predictions["feature_method"] = feature_processing_config["feature_method"]
        fold_predictions["y_true"] = y_fold_test.values
        fold_predictions["y_proba"] = y_proba
        fold_predictions["y_pred"] = (y_proba >= threshold).astype(int)

        prediction_rows.append(fold_predictions)

    fold_metrics_df = pd.DataFrame(fold_rows)

    if prediction_rows:
        oof_predictions_df = pd.concat(prediction_rows, ignore_index=True)
    else:
        oof_predictions_df = pd.DataFrame()

    return fold_metrics_df, oof_predictions_df

In [ ]:
# =========================
# 12. Smoke test CPCV with feature processing
# =========================

first_tb_config_name = list(datasets_by_config.keys())[0]
first_rf_config = RF_CONFIGS[0]

# Try "none", then "corr_cluster", then "pca"
first_feature_processing_config = FEATURE_PROCESSING_CONFIGS[1]

print(f"Triple-barrier config: {first_tb_config_name}")
print(f"RF config: {first_rf_config['model_name']}")
print(f"Feature method: {first_feature_processing_config['feature_method']}")

data = datasets_by_config[first_tb_config_name]

rf_model = make_random_forest(first_rf_config["model_params"])

fold_metrics_df, oof_predictions_df = run_cpcv_for_model(
    model=rf_model,
    train_df=data["train_df"],
    feature_cols=data["feature_cols"],
    target_col=TARGET_COL,
    feature_processing_config=first_feature_processing_config,
    n_blocks=10,
    n_test_blocks=2,
    embargo_size=data["num_days"],
    threshold=0.5,
)

display(fold_metrics_df.head())
display(oof_predictions_df.head())

print("Number of CPCV folds:", len(fold_metrics_df))
print("Mean AUC:", fold_metrics_df["auc"].mean())
print("Median AUC:", fold_metrics_df["auc"].median())
print("Mean F1:", fold_metrics_df["f1"].mean())
print("Average processed features:", fold_metrics_df["n_processed_features"].mean())

In [ ]:
# =========================
# 13. Run CPCV grid search
# =========================

def summarise_fold_metrics(
    fold_metrics_df: pd.DataFrame,
    tb_config_name: str,
    model_name: str,
    model_params: dict,
    feature_processing_config: dict,
    num_days: int | None,
) -> dict:
    """
    Summarises CPCV fold metrics for one TB config + model config + feature method.
    """
    return {
        "tb_config_name": tb_config_name,
        "model_name": model_name,
        "feature_method": feature_processing_config["feature_method"],
        "feature_params": feature_processing_config.get("feature_params", {}),
        "num_days": num_days,
        "n_folds": len(fold_metrics_df),

        "mean_auc": fold_metrics_df["auc"].mean(),
        "median_auc": fold_metrics_df["auc"].median(),
        "std_auc": fold_metrics_df["auc"].std(),

        "mean_accuracy": fold_metrics_df["accuracy"].mean(),
        "mean_precision": fold_metrics_df["precision"].mean(),
        "mean_recall": fold_metrics_df["recall"].mean(),
        "mean_f1": fold_metrics_df["f1"].mean(),

        "mean_n_train": fold_metrics_df["n_train"].mean(),
        "mean_n_test": fold_metrics_df["n_test"].mean(),

        "mean_original_features": fold_metrics_df["n_original_features"].mean(),
        "mean_processed_features": fold_metrics_df["n_processed_features"].mean(),
        "mean_dropped_features": fold_metrics_df["n_dropped_features"].mean(),
        "mean_pca_components": fold_metrics_df["n_pca_components"].mean(),
        "mean_pca_explained_variance": fold_metrics_df["pca_explained_variance"].mean(),

        "model_params": model_params,
    }


def run_rf_cpcv_grid(
    datasets_by_config: dict,
    rf_configs: list[dict],
    feature_processing_configs: list[dict],
    n_blocks: int = CPCV_N_BLOCKS,
    n_test_blocks: int = CPCV_N_TEST_BLOCKS,
    threshold: float = 0.5,
) -> tuple[pd.DataFrame, dict]:
    """
    Runs CPCV for every:
    - triple-barrier config
    - Random Forest config
    - feature processing config
    """
    summary_rows = []
    detailed_results = {}

    total_runs = (
        len(datasets_by_config)
        * len(rf_configs)
        * len(feature_processing_configs)
    )

    run_counter = 0

    for tb_config_name, data in datasets_by_config.items():
        train_df = data["train_df"]
        feature_cols = data["feature_cols"]
        num_days = data["num_days"]

        for rf_config in rf_configs:
            for feature_processing_config in feature_processing_configs:
                run_counter += 1

                model_name = rf_config["model_name"]
                model_params = rf_config["model_params"]
                feature_method = feature_processing_config["feature_method"]

                print("=" * 100)
                print(f"Run {run_counter}/{total_runs}")
                print(f"Triple-barrier config: {tb_config_name}")
                print(f"RF config: {model_name}")
                print(f"Feature method: {feature_method}")
                print(f"Train rows: {len(train_df)}")
                print(f"Original features: {len(feature_cols)}")
                print(f"Embargo size: {num_days}")

                model = make_random_forest(model_params)

                fold_metrics_df, oof_predictions_df = run_cpcv_for_model(
                    model=model,
                    train_df=train_df,
                    feature_cols=feature_cols,
                    target_col=TARGET_COL,
                    feature_processing_config=feature_processing_config,
                    n_blocks=n_blocks,
                    n_test_blocks=n_test_blocks,
                    embargo_size=num_days,
                    threshold=threshold,
                )

                result_key = f"{tb_config_name}__{model_name}__{feature_method}"

                detailed_results[result_key] = {
                    "tb_config_name": tb_config_name,
                    "model_name": model_name,
                    "model_params": model_params,
                    "feature_processing_config": feature_processing_config,
                    "fold_metrics": fold_metrics_df,
                    "oof_predictions": oof_predictions_df,
                }

                summary = summarise_fold_metrics(
                    fold_metrics_df=fold_metrics_df,
                    tb_config_name=tb_config_name,
                    model_name=model_name,
                    model_params=model_params,
                    feature_processing_config=feature_processing_config,
                    num_days=num_days,
                )

                summary_rows.append(summary)

                print(f"Mean AUC:   {summary['mean_auc']:.4f}")
                print(f"Median AUC: {summary['median_auc']:.4f}")
                print(f"Mean F1:    {summary['mean_f1']:.4f}")
                print(f"Mean processed features: {summary['mean_processed_features']:.1f}")

    summary_df = pd.DataFrame(summary_rows)

    summary_df = summary_df.sort_values(
        by=["mean_auc", "median_auc", "mean_f1"],
        ascending=False,
    ).reset_index(drop=True)

    return summary_df, detailed_results

In [ ]:
rf_cpcv_summary, rf_detailed_results = run_rf_cpcv_grid(
    datasets_by_config=datasets_by_config,
    rf_configs=RF_CONFIGS,
    feature_processing_configs=FEATURE_PROCESSING_CONFIGS,
    n_blocks=10,
    n_test_blocks=2,
    threshold=0.5,
)

display(rf_cpcv_summary)

In [ ]:
cols_to_show = [
    "tb_config_name",
    "model_name",
    "feature_method",
    "num_days",
    "n_folds",
    "mean_auc",
    "median_auc",
    "std_auc",
    "mean_precision",
    "mean_recall",
    "mean_f1",
    "mean_accuracy",
    "mean_original_features",
    "mean_processed_features",
    "mean_dropped_features",
    "mean_pca_components",
    "mean_pca_explained_variance",
]

top_rf_configs = rf_cpcv_summary[cols_to_show].head(10)

display(top_rf_configs)

In [ ]:
MODEL_OUTPUT_DIR = Path("../../data/model_outputs/random_forest")
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RF_CPCV_SUMMARY_PATH = MODEL_OUTPUT_DIR / f"{TICKER}_rf_cpcv_summary.csv"

rf_cpcv_summary_to_save = rf_cpcv_summary.copy()
rf_cpcv_summary_to_save["model_params"] = rf_cpcv_summary_to_save["model_params"].apply(json.dumps)
rf_cpcv_summary_to_save["feature_params"] = rf_cpcv_summary_to_save["feature_params"].apply(json.dumps)

rf_cpcv_summary_to_save.to_csv(RF_CPCV_SUMMARY_PATH, index=False)

print(f"Saved RF CPCV summary to: {RF_CPCV_SUMMARY_PATH}")

In [ ]:
# =========================
# 17. Select best RF config
# =========================

best_row = rf_cpcv_summary.iloc[0]

BEST_TB_CONFIG_NAME = best_row["tb_config_name"]
BEST_MODEL_NAME = best_row["model_name"]
BEST_MODEL_PARAMS = best_row["model_params"]

print("Best triple-barrier config:", BEST_TB_CONFIG_NAME)
print("Best RF config:", BEST_MODEL_NAME)
print("Best mean AUC:", best_row["mean_auc"])
print("Best median AUC:", best_row["median_auc"])
print("Best model params:")
print(BEST_MODEL_PARAMS)